# 🛒 Proyecto PGAD 2026-1
## Notebook 1 — Etapas 1 y 2: Carga, Limpieza y SQL

**Flujo de trabajo:**
1. Cargar y limpiar los archivos CSV (`transactions`, `customers`, `products`)
2. Unir las tablas y hacer una visualización exploratoria inicial
3. Exportar los datos a SQL y generar el archivo `.sql` de entrega

> Los archivos CSV se descargan automáticamente desde el repositorio de GitHub. No es necesario subir archivos manualmente.

## ⚙️ Etapa 1 — Carga de Datos desde GitHub y Limpieza

In [24]:
import pandas as pd
import numpy as np
import os
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from google.colab import data_table

# ── Clonar repositorio ─────────────────────────────────────────────────────────
REPO_URL = 'https://github.com/JCAMACHO05/Proyecto-PGAD.git'
REPO_DIR = '/content/Proyecto-PGAD'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

# ── Detectar automáticamente dónde están los CSV ──────────────────────────────
def encontrar_csv(base_dir, nombre):
    """Busca un CSV en la raíz y subcarpetas del repo."""
    for root, dirs, files in os.walk(base_dir):
        if nombre in files:
            return root
    retu

Already up to date.


In [25]:
import os
import pandas as pd

PATH = "/content/Proyecto-PGAD"  # ajusta esta ruta

def cargar_y_limpiar(nombre_archivo):
    """Carga un CSV desde el repo clonado, elimina duplicados e imputa nulos
    (mediana para numéricos, 'N/A' para texto). Retorna None si no existe."""

    ruta = os.path.join(PATH, nombre_archivo)

    if not os.path.exists(ruta):
        print(f"⚠️  No se encontró: {nombre_archivo} en {ruta}")
        return None

    df = pd.read_csv(ruta)
    antes = len(df)

    df = df.drop_duplicates()
    despues = len(df)

    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna('N/A')

    nulos_restantes = df.isnull().sum().sum()

    print(f"✅ {nombre_archivo}: {despues} filas ({antes - despues} duplicados eliminados, {nulos_restantes} nulos restantes)")

    return df


# ── Carga de archivos ─────────────────────────────────────────
df_trans = cargar_y_limpiar('transactions.csv')
df_cust  = cargar_y_limpiar('customers.csv')
df_prod  = cargar_y_limpiar('products.csv')


✅ transactions.csv: 1069 filas (0 duplicados eliminados, 0 nulos restantes)
✅ customers.csv: 286 filas (0 duplicados eliminados, 0 nulos restantes)
✅ products.csv: 66 filas (0 duplicados eliminados, 0 nulos restantes)


In [26]:
# ── Resumen estadístico de cada tabla ─────────────────────────────────────────
print("=" * 60)
print("TRANSACCIONES — shape:", df_trans.shape)
display(df_trans.describe(include='all').T)

print("\n" + "=" * 60)
print("CLIENTES — shape:", df_cust.shape)
display(df_cust.describe(include='all').T)

print("\n" + "=" * 60)
print("PRODUCTOS — shape:", df_prod.shape)
display(df_prod.describe(include='all').T)


TRANSACCIONES — shape: (1069, 14)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
transaction_id,1069.0,NaN,NaN,NaN,535.0,308.738023,1.0,268.0,535.0,802.0,1069.0
customer_id,1069.0,NaN,NaN,NaN,140.067353,81.836832,1.0,69.0,140.0,214.0,286.0
product_id,1069.0,NaN,NaN,NaN,33.27783,19.179091,1.0,16.0,33.0,50.0,66.0
transaction_date,1069,344,2025-10-16,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
channel,1069,4,Store,701,NaN,NaN,NaN,NaN,NaN,NaN,NaN
quantity,1069.0,NaN,NaN,NaN,4.212348,2.439684,1.0,2.0,4.0,6.0,22.0
unit_price,1069.0,NaN,NaN,NaN,162.930075,200.69287,3.99,16.68,65.43,249.39,826.99
discount_pct,1069.0,NaN,NaN,NaN,0.104337,0.065475,0.0,0.053,0.103,0.148,0.331
promo_flag,1069.0,NaN,NaN,NaN,0.292797,0.455259,0.0,0.0,0.0,1.0,1.0
payment_method,1069,6,Digital Wallet,665,NaN,NaN,NaN,NaN,NaN,NaN,NaN



CLIENTES — shape: (286, 8)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customer_id,286.0,NaN,NaN,NaN,143.5,82.7053,1.0,72.25,143.5,214.75,286.0
age,286.0,NaN,NaN,NaN,35.842657,11.171379,18.0,28.0,36.0,43.0,70.0
gender,286,4,F,136,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,286,9,Bucaramanga,121,NaN,NaN,NaN,NaN,NaN,NaN,NaN
membership_level,286,4,Basic,160,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_tenure_months,286.0,NaN,NaN,NaN,21.548951,13.72727,1.0,11.0,20.0,31.75,63.0
loyalty_score,286.0,NaN,NaN,NaN,54.366434,17.584963,0.0,43.6525,53.56,65.6625,100.0
estimated_income,286.0,NaN,NaN,NaN,3240.120385,1266.891487,700.0,2378.6825,3233.25,4095.4075,7207.58



PRODUCTOS — shape: (66, 5)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
product_id,66.0,NaN,NaN,NaN,33.5,19.196354,1.0,17.25,33.5,49.75,66.0
category,66,5,Electronics,21,NaN,NaN,NaN,NaN,NaN,NaN,NaN
brand,66,9,Pulse,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN
supplier_region,66,7,Local,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
base_price,66.0,NaN,NaN,NaN,169.139848,199.082721,4.41,17.325,74.805,249.3375,717.07


In [27]:
# ── Unión de tablas y visualización inicial ────────────────────────────────────
df_trans['transaction_date'] = pd.to_datetime(df_trans['transaction_date'])

df_merged = (
    df_trans
    .merge(df_cust[['customer_id', 'city', 'membership_level']], on='customer_id', how='left')
    .merge(df_prod[['product_id', 'category', 'brand']], on='product_id', how='left')
)

data_table.enable_dataframe_formatter()
print(f"✅ Tabla unida: {df_merged.shape[0]} filas × {df_merged.shape[1]} columnas")
display(df_merged.head(10))


✅ Tabla unida: 1069 filas × 18 columnas


,transaction_id,customer_id,product_id,transaction_date,channel,quantity,unit_price,discount_pct,promo_flag,payment_method,days_since_last_purchase,prior_purchases,total_amount,repeat_purchase_next_month,city,membership_level,category,brand
0,703,176,3,2025-01-19,Web,5,16.07,0.166,1,Digital Wallet,33,8,94.27,1,N/A,Basic,Groceries,Zenit
1,915,182,11,2025-03-14,Web,7,336.58,0.090,0,Digital Wallet,21,10,2363.20,0,Bucaramanga,Silver,Electronics,Aurum
2,931,152,37,2025-11-05,App,1,5.66,0.261,0,Digital Wallet,50,8,10.51,1,Bucaramanga,Platinum,Groceries,N/A
3,814,239,30,2025-05-07,N/A,5,18.92,0.203,0,Digital Wallet,26,5,75.95,1,Bucaramanga,N/A,Beauty,Viva
4,449,80,62,2025-06-22,Store,7,40.57,0.000,1,Digital Wallet,45,4,367.01,0,Bogota,Silver,Beauty,Nova
5,766,227,50,2025-09-03,Store,3,447.57,0.029,0,Credit Card,57,13,1160.54,0,Medellin,N/A,Electronics,Nova
6,381,124,11,2025-08-07,App,5,276.09,0.085,1,Credit Card,62,10,1290.28,0,Cartagena,Silver,Electronics,Aurum
7,654,93,26,2025-12-03,Store,5,58.33,0.031,0,Digital Wallet,26,11,266.75,0,Bucaramanga,Basic,Beauty,Aurum
8,936,87,65,2025-02-03,Store,7,348.19,0.071,0,Transfer,26,13,2402.23,0,Bogota,Basic,Electronics,Pulse
9,59,82,60,2025-05-21,App,3,44.20,0.145,0,Digital Wallet,29,7,122.66,1,Bogota,Basic,Sports,Pulse


In [28]:
# ── Visualizaciones Exploratorias — Etapa 1 ────────────────────────────────────

# Panel 1: Distribución de ventas por categoría + canal de venta
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'pie'}, {'type': 'bar'}]],
    subplot_titles=[
        'Distribución de Ventas por Categoría',
        'Total de Ventas por Canal'
    ]
)

cat_sales = df_merged.groupby('category')['total_amount'].sum().reset_index()
fig.add_trace(
    go.Pie(
        labels=cat_sales['category'], values=cat_sales['total_amount'],
        hole=0.4, marker_colors=px.colors.qualitative.Pastel,
        textinfo='label+percent'
    ),
    row=1, col=1
)

ch_sales = df_merged.groupby('channel')['total_amount'].sum().reset_index().sort_values('total_amount', ascending=False)
fig.add_trace(
    go.Bar(
        x=ch_sales['channel'], y=ch_sales['total_amount'],
        marker_color=px.colors.qualitative.Pastel[:len(ch_sales)],
        text=ch_sales['total_amount'].map('${:,.0f}'.format),
        textposition='outside'
    ),
    row=1, col=2
)

fig.update_layout(
    title_text='📊 Exploración Inicial — Ventas por Categoría y Canal',
    height=450, showlegend=False, title_font_size=16
)
fig.show()

# Panel 2: Nivel de membresía + distribución del monto
fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Clientes por Nivel de Membresía',
        'Distribución del Monto Total'
    ]
)

memb = df_merged['membership_level'].value_counts().reset_index()
memb.columns = ['membership_level', 'count']
fig2.add_trace(
    go.Bar(
        x=memb['membership_level'], y=memb['count'],
        marker_color=px.colors.qualitative.Safe[:len(memb)],
        text=memb['count'], textposition='outside'
    ),
    row=1, col=1
)

fig2.add_trace(
    go.Histogram(
        x=df_merged['total_amount'], nbinsx=40,
        marker_color='steelblue', opacity=0.75, name='total_amount'
    ),
    row=1, col=2
)

promedio = df_merged['total_amount'].mean()
fig2.add_vline(x=promedio, line_dash='dash', line_color='red',
               annotation_text=f'Promedio: ${promedio:.0f}',
               annotation_position='top right', row=1, col=2)

fig2.update_layout(
    title_text='👥 Clientes y Distribución de Montos',
    height=430, showlegend=False, title_font_size=16
)
fig2.show()

df_filtrado = df_merged[df_merged['total_amount'] > promedio].copy()
print(f"\nTransacciones sobre el promedio (${promedio:.2f}): {len(df_filtrado)} registros ({len(df_filtrado)/len(df_merged)*100:.1f}%)")
display(df_filtrado.head(20))


Transacciones sobre el promedio ($687.08): 310 registros (29.0%)


,transaction_id,customer_id,product_id,transaction_date,channel,quantity,unit_price,discount_pct,promo_flag,payment_method,days_since_last_purchase,prior_purchases,total_amount,repeat_purchase_next_month,city,membership_level,category,brand
1,915,182,11,2025-03-14,Web,7,336.58,0.090,0,Digital Wallet,21,10,2363.20,0,Bucaramanga,Silver,Electronics,Aurum
5,766,227,50,2025-09-03,Store,3,447.57,0.029,0,Credit Card,57,13,1160.54,0,Medellin,N/A,Electronics,Nova
6,381,124,11,2025-08-07,App,5,276.09,0.085,1,Credit Card,62,10,1290.28,0,Cartagena,Silver,Electronics,Aurum
8,936,87,65,2025-02-03,Store,7,348.19,0.071,0,Transfer,26,13,2402.23,0,Bogota,Basic,Electronics,Pulse
12,789,194,14,2025-02-01,Store,7,474.63,0.061,0,Digital Wallet,22,4,3734.60,0,Bogota,Silver,Electronics,Pulse
16,98,53,50,2025-03-10,Store,3,415.30,0.093,1,Digital Wallet,53,7,1367.48,0,cali,Silver,Electronics,Nova
18,900,176,1,2025-03-28,Store,3,547.67,0.001,0,Digital Wallet,19,10,1813.95,0,N/A,Basic,Electronics,Pulse
19,656,65,57,2025-04-30,Web,5,203.53,0.102,1,Digital Wallet,52,10,1114.97,0,Bogota,Platinum,Home,Nova
20,746,238,25,2025-07-04,Store,7,156.30,0.063,0,Debit Card,18,6,1017.79,0,Bogota,Basic,Sports,Orion
24,1009,22,59,2025-07-20,App,6,289.76,0.036,0,Digital Wallet,57,5,1897.55,0,Bogota,Silver,Electronics,Orion


---
## 🗄️ Etapa 2 — SQL: Creación de Base de Datos y Exportación

In [29]:
import sqlite3

# ── Crear base de datos en memoria y cargar tablas ────────────────────────────
conn = sqlite3.connect(':memory:')

df_trans.to_sql('transactions', conn, index=False, if_exists='replace')
df_cust.to_sql('customers',    conn, index=False, if_exists='replace')
df_prod.to_sql('products',     conn, index=False, if_exists='replace')

# ── Consulta de integración con filtro sobre el promedio ──────────────────────
query = """
SELECT
    t.*,
    c.city,
    c.membership_level,
    p.category,
    p.brand
FROM transactions t
LEFT JOIN customers c ON t.customer_id = c.customer_id
LEFT JOIN products  p ON t.product_id  = p.product_id
WHERE t.total_amount > (SELECT AVG(total_amount) FROM transactions)
ORDER BY t.total_amount DESC
"""

df_resultado_sql = pd.read_sql_query(query, conn)
print(f"✅ Consulta SQL ejecutada: {len(df_resultado_sql)} filas retornadas")
display(df_resultado_sql.head(10))


✅ Consulta SQL ejecutada: 310 filas retornadas


,transaction_id,customer_id,product_id,transaction_date,channel,quantity,unit_price,discount_pct,promo_flag,payment_method,days_since_last_purchase,prior_purchases,total_amount,repeat_purchase_next_month,city,membership_level,category,brand
0,976,87,50,2025-12-09 00:00:00,App,11,428.80,0.031,0,Digital Wallet,24,7,10445.73,0,Bogota,Basic,Electronics,Nova
1,851,130,6,2025-10-12 00:00:00,Store,22,226.37,0.231,0,Digital Wallet,31,14,8552.18,0,Bogota,Silver,Sports,Orion
2,937,125,35,2025-07-17 00:00:00,Store,15,258.32,0.037,0,Credit Card,29,9,7677.50,0,Bogota,Basic,Sports,Zenit
3,438,169,63,2025-04-20 00:00:00,App,7,746.44,0.072,1,Digital Wallet,51,14,6721.73,0,Bucaramanga,Basic,Electronics,Pulse
4,1034,202,63,2025-04-11 00:00:00,Web,7,787.61,0.000,0,Debit Card,26,16,6168.67,0,Bucaramanga,Basic,Electronics,Pulse
5,1026,68,33,2025-05-24 00:00:00,Store,7,698.83,0.000,0,Debit Card,46,11,6108.68,0,N/A,Basic,Electronics,Aurum
6,921,51,63,2025-02-15 00:00:00,App,6,715.93,0.040,1,Digital Wallet,45,7,5407.14,0,Bucaramanga,Basic,Electronics,Pulse
7,999,276,63,2025-01-18 00:00:00,Store,7,664.18,0.060,1,Debit Card,32,7,5392.73,0,Bucaramanga,Platinum,Electronics,Pulse
8,550,107,61,2025-01-07 00:00:00,Store,7,754.92,0.105,1,Digital Wallet,47,13,5164.77,0,Medellin,Basic,Electronics,Nova
9,906,165,47,2025-09-22 00:00:00,Store,15,217.11,0.110,1,Digital Wallet,24,12,5019.72,0,Bucaramanga,Basic,Electronics,Atlas


In [30]:
# ── Visualización del resultado SQL ───────────────────────────────────────────
top_brands = (
    df_resultado_sql
    .groupby('brand')['total_amount']
    .sum()
    .nlargest(10)
    .reset_index()
)

fig3 = px.bar(
    top_brands, x='total_amount', y='brand', orientation='h',
    title='🏆 Top 10 Marcas — Ventas sobre el Promedio (resultado SQL)',
    labels={'total_amount': 'Ventas Totales ($)', 'brand': 'Marca'},
    color='total_amount',
    color_continuous_scale='Blues',
    text=top_brands['total_amount'].map('${:,.0f}'.format)
)
fig3.update_traces(textposition='outside')
fig3.update_layout(height=400, coloraxis_showscale=False, yaxis={'categoryorder': 'total ascending'})
fig3.show()


In [31]:
from google.colab import files

# ── Generar archivo .sql de entrega ───────────────────────────────────────────
sql_file = 'Tabla_unificada.sql'

with open(sql_file, 'w', encoding='utf-8') as f:
    f.write('-- ============================================================\n')
    f.write('-- PROYECTO PGAD 2026-1: ETAPA 2 (SQL)\n')
    f.write('-- Esquema de base de datos y carga de registros\n')
    f.write('-- ============================================================\n\n')

    f.write('-- DDL: Creación de tablas\n')
    f.write(
        'CREATE TABLE IF NOT EXISTS customers (\n'
        '    customer_id              INTEGER PRIMARY KEY,\n'
        '    age                      FLOAT,\n'
        '    gender                   TEXT,\n'
        '    city                     TEXT,\n'
        '    membership_level         TEXT,\n'
        '    customer_tenure_months   INTEGER,\n'
        '    loyalty_score            FLOAT,\n'
        '    estimated_income         FLOAT\n'
        ');\n\n'
    )
    f.write(
        'CREATE TABLE IF NOT EXISTS products (\n'
        '    product_id      INTEGER PRIMARY KEY,\n'
        '    category        TEXT,\n'
        '    brand           TEXT,\n'
        '    supplier_region TEXT,\n'
        '    base_price      FLOAT\n'
        ');\n\n'
    )
    f.write(
        'CREATE TABLE IF NOT EXISTS transactions (\n'
        '    transaction_id               INTEGER PRIMARY KEY,\n'
        '    customer_id                  INTEGER,\n'
        '    product_id                   INTEGER,\n'
        '    transaction_date             DATE,\n'
        '    channel                      TEXT,\n'
        '    quantity                     INTEGER,\n'
        '    unit_price                   FLOAT,\n'
        '    discount_pct                 FLOAT,\n'
        '    promo_flag                   INTEGER,\n'
        '    payment_method               TEXT,\n'
        '    days_since_last_purchase     INTEGER,\n'
        '    prior_purchases              INTEGER,\n'
        '    total_amount                 FLOAT,\n'
        '    repeat_purchase_next_month   INTEGER,\n'
        '    FOREIGN KEY(customer_id) REFERENCES customers(customer_id),\n'
        '    FOREIGN KEY(product_id)  REFERENCES products(product_id)\n'
        ');\n\n'
    )

    f.write('-- DML: Carga de datos — CUSTOMERS\n')
    for _, row in df_cust.iterrows():
        f.write(
            f"INSERT INTO customers VALUES ({row['customer_id']}, {row['age']}, "
            f"'{row['gender']}', '{row['city']}', '{row['membership_level']}', "
            f"{row['customer_tenure_months']}, {row['loyalty_score']}, {row['estimated_income']});\n"
        )

    f.write('\n-- DML: Carga de datos — PRODUCTS\n')
    for _, row in df_prod.iterrows():
        f.write(
            f"INSERT INTO products VALUES ({row['product_id']}, '{row['category']}', "
            f"'{row['brand']}', '{row['supplier_region']}', {row['base_price']});\n"
        )

    f.write('\n-- DML: Carga de datos — TRANSACTIONS\n')
    for _, row in df_trans.iterrows():
        f.write(
            f"INSERT INTO transactions VALUES ({row['transaction_id']}, {row['customer_id']}, "
            f"{row['product_id']}, '{row['transaction_date']}', '{row['channel']}', "
            f"{row['quantity']}, {row['unit_price']}, {row['discount_pct']}, "
            f"{row['promo_flag']}, '{row['payment_method']}', {row['days_since_last_purchase']}, "
            f"{row['prior_purchases']}, {row['total_amount']}, {row['repeat_purchase_next_month']});\n"
        )

print(f"✅ Archivo '{sql_file}' generado correctamente")
files.download(sql_file)

✅ Archivo 'Tabla_unificada.sql' generado correctamente


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>